### 0. Settings


In [ ]:
from datetime import datetime, timedelta
# Date defaults
START_DATE = "2025-12-01"
DATE = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d") # yesterday completed data

### This file is used to fetch data from Singapore NSE
- Dataset (.csv) downloaded are
    - day
    - 30 day
    - Month (Dec - Jun)

In [3]:
from io import StringIO
from datetime import datetime
import pandas as pd
import requests

def fetch_nems_day(target_date):
    """
    Fetches a single day of Singapore USEP and Demand Forecast.

    Args:
        - target_date: DATE, which is yesterday's date (YYYY-MM-DD format)

    Returns:
        - df: DataFrame
    """
    url = "https://www.nems.emcsg.com/api/sitecore/DataSync/DataDownload"
    # (Optional) Manually downloaded from https://www.nems.emcsg.com/NEMS-Market-Trading-Reports.

    params = {"value": 10, "fromDate": target_date, "toDate": target_date, "tpcValue": 1}

    # Crucial step: Mirror a real browser to bypass security blocks
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"Requesting data for {target_date}...")
    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
        # Check if the site returned an error page disguised as success
        if "No data found" in response.text or "Error" in response.text:
            print(f"No market data available yet for {target_date}.")
            return None

        # Convert raw text string directly into a Pandas Dataframe
        df = pd.read_csv(StringIO(response.text))
        print("Data downloaded successfully!")
        return df
    else:
        print(f"Request failed with status code: {response.status_code}")
        return None

DATE = "2026-06-19"
a_df = fetch_nems_day(DATE)

Requesting data for 2026-06-19...
Data downloaded successfully!


In [ ]:
# OPTIONAL to run since month.csv is already in tools folder.
import time
 # 1. Generate a list of dates for the last 6 months
start_date = "2025-12-18"
end_date = "2026-06-17"
date_list = pd.date_range(start=start_date, end=end_date).strftime("%Y-%m-%d")

url = "https://www.nems.emcsg.com/api/sitecore/DataSync/DataDownload"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36..."
}

all_days_data = []

print(f"Starting automated download for {len(date_list)} days...")

# 2. Loop through every single day automatically
for target_date in date_list:
    params = {
        "value": 10,
        "fromDate": target_date,
        "toDate": target_date,
        "tpcValue": 1,
    }

    try:
        response = requests.get(url, params=params, headers=headers)

        if (
            response.status_code == 200
            and "No data found" not in response.text
        ):
            day_df = pd.read_csv(StringIO(response.text))
            all_days_data.append(day_df)
            print(f"  Successfully grabbed: {target_date}")
        else:
            print(f"Skipped or no data for: {target_date}")

    except Exception as e:
        print(f"Error on {target_date}: {e}")

    # Crucial: Pause for 0.5 seconds between requests so the server doesn't block you
    time.sleep(0.5)

# 3. Combine all 30 days into one master DataFrame
if all_days_data:
    a_df = pd.concat(all_days_data, ignore_index=True)

    # Save to .csv file.
    a_df.to_csv("7_month.csv", index=False)
    print(f"Done! Combined dataset contains {len(a_df)} rows and is saved to disk.")
else:
    print("No data was collected.")

Starting automated download for 182 days...
  Successfully grabbed: 2025-12-18
  Successfully grabbed: 2025-12-19
  Successfully grabbed: 2025-12-20
  Successfully grabbed: 2025-12-21
  Successfully grabbed: 2025-12-22
  Successfully grabbed: 2025-12-23
  Successfully grabbed: 2025-12-24
  Successfully grabbed: 2025-12-25
  Successfully grabbed: 2025-12-26
  Successfully grabbed: 2025-12-27
  Successfully grabbed: 2025-12-28
  Successfully grabbed: 2025-12-29
  Successfully grabbed: 2025-12-30
  Successfully grabbed: 2025-12-31
  Successfully grabbed: 2026-01-01
  Successfully grabbed: 2026-01-02
  Successfully grabbed: 2026-01-03
  Successfully grabbed: 2026-01-04
  Successfully grabbed: 2026-01-05
  Successfully grabbed: 2026-01-06
  Successfully grabbed: 2026-01-07
  Successfully grabbed: 2026-01-08
  Successfully grabbed: 2026-01-09
  Successfully grabbed: 2026-01-10
  Successfully grabbed: 2026-01-11
  Successfully grabbed: 2026-01-12
  Successfully grabbed: 2026-01-13
  Successfu